In [0]:
%sql
-- Días sin datos ADEMÁS DE los primeros días de ENERO 2025
WITH rango AS (
  SELECT MIN(date) AS ini, MAX(date) AS fin
  FROM workspace.default.silver_ventas
),
calendario AS (
  SELECT explode(sequence(ini, fin, INTERVAL 1 DAY)) AS date
  FROM rango
)
SELECT c.date, date_format(c.date, 'EEEE') AS dia_semana
FROM calendario c
LEFT ANTI JOIN (SELECT DISTINCT date FROM workspace.default.silver_ventas) s
  ON c.date = s.date
ORDER BY c.date

In [0]:
%sql
-- Contabilización de días sin datos
WITH calendario AS (
  SELECT explode(sequence(DATE'2025-01-01', DATE'2026-09-27', INTERVAL 1 DAY)) AS date
),
con_datos AS (
  SELECT DISTINCT date FROM workspace.default.silver_ventas
)
SELECT COUNT(*) AS dias_calendario,
       COUNT(d.date) AS dias_con_datos,
       COUNT(*) - COUNT(d.date) AS dias_sin_datos,
       COUNT_IF(d.date IS NULL AND c.date < DATE'2025-01-22') AS vacios_enero,
       COUNT_IF(d.date IS NULL AND c.date >= DATE'2025-01-22') AS huecos
FROM calendario c
LEFT JOIN con_datos d ON c.date = d.date

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_ventas AS
SELECT s.* EXCEPT (quantity, _source_file, _n_cols,
                   menu_price, total, item_discount, total_discount, tax,
                   item_voided, order_voided),
       try_cast(s.menu_price AS DECIMAL(12,4)) AS menu_price,
       try_cast(s.total AS DECIMAL(12,4)) AS total,
       try_cast(s.item_discount AS DECIMAL(12,4)) AS item_discount,
       try_cast(s.total_discount AS DECIMAL(12,4)) AS total_discount,
       try_cast(s.tax AS DECIMAL(12,4)) AS tax,
       try_cast(s.item_voided AS DECIMAL(12,4)) AS item_voided,
       try_cast(s.order_voided AS DECIMAL(12,4)) AS order_voided,
       timestampdiff(MINUTE, s.begin_time, s.end_time) AS duracion_min
FROM workspace.default.silver_ventas s

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_ventas AS
SELECT * EXCEPT (tiempo_espera_segundos)
FROM workspace.default.silver_ventas

In [0]:
%sql
-- .head() de la tabla final
SELECT * FROM workspace.default.silver_ventas LIMIT 10;

In [0]:
%sql
SELECT
  count_if(store IS NULL)            AS store,
  count_if(pos_code IS NULL)         AS pos_code,
  count_if(check_number IS NULL)     AS check_number,
  count_if(menu_item IS NULL)        AS menu_item,
  count_if(menu_price IS NULL)       AS menu_price,
  count_if(total IS NULL)            AS total,
  count_if(item_discount IS NULL)    AS item_discount,
  count_if(total_discount IS NULL)   AS total_discount,
  count_if(tax IS NULL)              AS tax,
  count_if(item_voided IS NULL)      AS item_voided,
  count_if(order_voided IS NULL)     AS order_voided,
  count_if(transaction_type IS NULL) AS transaction_type,
  count_if(tender_type IS NULL)      AS tender_type,
  count_if(report_group IS NULL)     AS report_group,
  count_if(pop IS NULL)              AS pop,
  count_if(date IS NULL)             AS date,
  count_if(begin_time IS NULL)       AS begin_time,
  count_if(end_time IS NULL)         AS end_time
FROM workspace.default.silver_ventas;

In [0]:
%sql
-- limpieza de nulls 
DECLARE OR REPLACE VARIABLE filas_antes BIGINT;
SET VAR filas_antes = (SELECT count(*) FROM workspace.default.silver_ventas);

CREATE OR REPLACE TABLE workspace.default.silver_ventas AS
SELECT *
FROM workspace.default.silver_ventas
WHERE store IS NOT NULL
  AND pos_code IS NOT NULL
  AND check_number IS NOT NULL
  AND menu_price IS NOT NULL
  AND total IS NOT NULL
  AND item_discount IS NOT NULL
  AND total_discount IS NOT NULL
  AND tax IS NOT NULL
  AND item_voided IS NOT NULL
  AND order_voided IS NOT NULL
  AND transaction_type IS NOT NULL
  AND pop IS NOT NULL
  AND date IS NOT NULL
  AND begin_time IS NOT NULL
  AND end_time IS NOT NULL
  AND duracion_min IS NOT NULL;

SELECT
  filas_antes,
  count(*)               AS filas_despues,
  filas_antes - count(*) AS filas_quitadas
FROM workspace.default.silver_ventas;

In [0]:
%sql
SELECT tender_type, count(*) AS n,
       round(100 * count(*) / sum(count(*)) OVER (), 3) AS pct,
       round(100 * sum(count(*)) OVER (ORDER BY count(*) DESC) / sum(count(*)) OVER (), 2) AS pct_acum
FROM workspace.default.silver_ventas
GROUP BY tender_type
ORDER BY n DESC;

In [0]:
%sql
-- Analisis de pop
SELECT pop, count(*) AS n,
       round(100 * count(*) / sum(count(*)) OVER (), 3) AS pct,
       round(100 * sum(count(*)) OVER (ORDER BY count(*) DESC) / sum(count(*)) OVER (), 2) AS pct_acum
FROM workspace.default.silver_ventas
GROUP BY pop
ORDER BY n DESC;

In [0]:
%sql
-- Se puede confirmar que se renombraron las variables
SELECT date_trunc('MONTH', date) AS mes,
       count_if(pop = 'To Go')      AS to_go,
       count_if(pop = 'Drive Thru') AS drive_thru,
       count_if(pop = 'Inside')     AS inside,
       count_if(pop = 'Dine-In')    AS dine_in,
       count_if(pop = 'Carry Out')  AS carry_out,
       count_if(pop = '3')          AS tres
FROM workspace.default.silver_ventas
GROUP BY ALL
ORDER BY mes;

In [0]:
%sql
-- Reducir categorías de pop
UPDATE workspace.default.silver_ventas
SET pop = CASE pop
            WHEN 'To Go'  THEN 'Drive Thru'
            WHEN 'Inside' THEN 'Dine-In'
            WHEN '3'      THEN 'Carry Out'
          END
WHERE pop IN ('To Go', 'Inside', '3');

In [0]:
%sql
-- Reducir categorías de tender_type
-- Reglas:
--   NULL → NULL (sin cambio)
--   Billetes puros o combinaciones de solo billetes → 'Cash'
--   Combinaciones con tarjeta (cash+tarjeta, dos tarjetas, billete+tarjeta) → 'Hybrid'
--   Tarjetas individuales → se quedan como están
UPDATE workspace.default.silver_ventas
SET tender_type = CASE
  -- Billetes individuales → Cash
  WHEN tender_type IN ('Cash', '$1', '$5', '$10', '$20', '$50') THEN 'Cash'
  -- Compuesto con al menos una tarjeta/plataforma → Hybrid
  WHEN contains(tender_type, ':') AND (
       tender_type LIKE '%Visa%'
    OR tender_type LIKE '%MasterCard%'
    OR tender_type LIKE '%Discover%'
    OR tender_type LIKE '%American Express%'
    OR tender_type LIKE '%OLO%'
    OR tender_type LIKE '%EZCater%'
    OR tender_type LIKE '%FB DoorDash%'
    OR tender_type LIKE '%Online Orders%'
    OR tender_type LIKE '%EOD Tender%'
  ) THEN 'Hybrid'
  -- Compuesto de solo billetes → Cash
  WHEN contains(tender_type, ':') THEN 'Cash'
  -- Tarjetas individuales → sin cambio
  ELSE tender_type
END
WHERE tender_type IS NOT NULL;

In [0]:
%sql
-- Verificación de tender_type después de la limpieza
SELECT tender_type, count(*) AS n,
       round(100 * count(*) / sum(count(*)) OVER (), 3) AS pct
FROM workspace.default.silver_ventas
GROUP BY tender_type
ORDER BY n DESC;

In [0]:
%sql
-- Crear tabla Gold a partir de la Silver ya limpia
CREATE OR REPLACE TABLE workspace.default.gold_ventas AS
SELECT *
FROM workspace.default.silver_ventas;